In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install face_recognition
!pip install deepface
!pip install opencv-python-headless
!pip install tensorflow
!pip install keras
!pip install numpy


In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install numpy
!pip install tensorflow
!pip install keras
!pip install opencv-python-headless
!pip install face_recognition
!pip install deepface


In [ ]:
import os

print("Текущая рабочая директория:", os.getcwd())

print("\nСодержимое рабочей директории:")
for item in os.listdir():
    print("-", item)

dataset_path = "/kaggle/input"

if os.path.exists(dataset_path):
    print(f"\nСодержимое папки {dataset_path}:")
    for folder in os.listdir(dataset_path):
        print("-", folder)
else:
    print(f"\nПапка {dataset_path} не найдена. Дatasets, возможно, не подключены.")

In [ ]:
import cv2
import face_recognition
import os
import urllib.request 
import numpy as np
from deepface import DeepFace
import time



In [ ]:

def run_face_recognition_pipeline(
    video_capture,
    out,
    input_video_path,
    known_face_encodings,
    known_face_names,
    use_cuda_resize=True,
    scale=0.25,
    print_every=10
):
   
    frame_num = 0
    faces_drawn = 0
    t0 = time.perf_counter()

    try:
        total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    except Exception:
        total_frames = -1

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        frame_num += 1

        t_frame_start = time.perf_counter()  

        if use_cuda_resize:
            try:
                gpu_frame = cv2.cuda_GpuMat()
                gpu_frame.upload(frame)
                gpu_small = cv2.cuda.resize(gpu_frame, (0, 0), fx=scale, fy=scale)
                small_frame = gpu_small.download()
            except Exception:
                small_frame = cv2.resize(frame, (0, 0), fx=scale, fy=scale)
        else:
            small_frame = cv2.resize(frame, (0, 0), fx=scale, fy=scale)

        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        inv = int(round(1.0 / scale))
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            name = "Unknown"
            if len(known_face_encodings) > 0:
                matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
                face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
                if len(face_distances) > 0:
                    best_match_index = int(np.argmin(face_distances))
                    if matches[best_match_index]:
                        name = known_face_names[best_match_index]

            top *= inv
            right *= inv
            bottom *= inv
            left *= inv

            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, name, (left + 6, bottom - 6),
                        cv2.FONT_HERSHEY_DUPLEX, 1.0, (255, 255, 255), 1)
            faces_drawn += 1

        out.write(frame)

        t_frame_end = time.perf_counter()
        ms_per_frame = (t_frame_end - t_frame_start) * 1000.0  

        if print_every and (frame_num % print_every == 0):
            h, w = frame.shape[:2]
            total_txt = total_frames if total_frames > 0 else "?"
            print(f"video 1/1 (frame {frame_num}/{total_txt}) [{input_video_path}]: {w}x{h} | {ms_per_frame:.1f} ms/frame")

    elapsed = time.perf_counter() - t0
    return {
        "frames_processed": frame_num,
        "faces_drawn": faces_drawn,
        "elapsed_sec": elapsed,
        "avg_ms_per_frame": (elapsed / frame_num * 1000.0) if frame_num > 0 else None,  # среднее время на кадр в мс [web:115][web:117]
        "scale": scale,
        "used_cuda_resize": bool(use_cuda_resize),
        "output_path": getattr(out, 'filename', 'output file')
    }


In [ ]:
known_faces_dir=r"/kaggle/input/face-models-data/face_model_videos/labels"
known_face_encodings=[]
known_face_names=[]


if not os.path.exists(known_faces_dir):
    print(f"Ошибка: Папка '{known_faces_dir}' не найдена.")
    exit()

for filename in os.listdir(known_faces_dir):
    filepath = os.path.join(known_faces_dir, filename)
    image = face_recognition.load_image_file(filepath)
    encodings = face_recognition.face_encodings(image)
    if encodings:
        known_face_encodings.append(encodings[0])
        known_face_names.append(os.path.splitext(filename)[0])
        print(f" - {filename} загружено как {os.path.splitext(filename)[0]}")
if not known_face_encodings:
    print("Критическая ошибка: Не найдено ни одного лица в папке 'known_faces'.")
    exit()

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV"
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "bad_light_output_FINAL.mp4")

if not os.path.exists(input_video_path):
    raise FileNotFoundError(f"Видео не найдено: {input_video_path}")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError("Ошибка: Не удалось открыть видео.")

frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video_capture.get(cv2.CAP_PROP_FPS)


fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

# bad

In [ ]:
summary = run_face_recognition_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,  #
    known_face_encodings=known_face_encodings,
    known_face_names=known_face_names,
    use_cuda_resize=True,   
    scale=0.25,              
    print_every=5
)

# good

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV"
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "good_light_output_FINAL.mp4")

if not os.path.exists(input_video_path):
    raise FileNotFoundError(f"Видео не найдено: {input_video_path}")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError("Ошибка: Не удалось открыть видео.")

frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video_capture.get(cv2.CAP_PROP_FPS)


fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

In [ ]:
summary = run_face_recognition_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,  
    known_face_encodings=known_face_encodings,
    known_face_names=known_face_names,
    use_cuda_resize=True,   
    scale=0.25,             
    print_every=5
)

# BLUE

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/blue_light.MOV"
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "blue_light_output_FINAL.mp4")

if not os.path.exists(input_video_path):
    raise FileNotFoundError(f"Видео не найдено: {input_video_path}")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError("Ошибка: Не удалось открыть видео.")

frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video_capture.get(cv2.CAP_PROP_FPS)


fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

In [ ]:
summary = run_face_recognition_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,  
    known_face_encodings=known_face_encodings,
    known_face_names=known_face_names,
    use_cuda_resize=True,   
    scale=0.25,              
    print_every=5
)

# RED

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV"
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "red_light_output_final.mp4")

if not os.path.exists(input_video_path):
    raise FileNotFoundError(f"Видео не найдено: {input_video_path}")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError("Ошибка: Не удалось открыть видео.")

frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video_capture.get(cv2.CAP_PROP_FPS)


fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

In [ ]:
summary = run_face_recognition_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path, 
    known_face_encodings=known_face_encodings,
    known_face_names=known_face_names,
    use_cuda_resize=True,   
    scale=0.25,             
    print_every=5
)

In [ ]:

def ensure_haarcascades(face_cascade_filename="haarcascade_frontalface_default.xml",
                        eye_cascade_filename="haarcascade_eye.xml"):
    face_cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/" + face_cascade_filename
    eye_cascade_url  = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/" + eye_cascade_filename

    if not os.path.exists(face_cascade_filename):
        print(f"Файл '{face_cascade_filename}' не найден, скачивает...")
        urllib.request.urlretrieve(face_cascade_url, face_cascade_filename)
        print("Файл скачан")

    if not os.path.exists(eye_cascade_filename):
        print(f"Файл '{eye_cascade_filename}' не найден, скачивает...")
        urllib.request.urlretrieve(eye_cascade_url, eye_cascade_filename)
        print("Файл скачан")

    face_cascade = cv2.CascadeClassifier(face_cascade_filename)
    eye_cascade  = cv2.CascadeClassifier(eye_cascade_filename)

    if face_cascade.empty() or eye_cascade.empty():
        raise RuntimeError("Не удалось загрузить каскады. Удалите .xml и повторите запуск.")

    return face_cascade, eye_cascade


def run_opencv_haarcascade_pipeline(
    video_capture,
    out,
    input_video_path,
    use_cuda_resize=True,
    scale=1.0,               
    print_every=10,
    face_cascade_filename="haarcascade_frontalface_default.xml",
    eye_cascade_filename="haarcascade_eye.xml",
    face_scaleFactor=1.1,
    face_minNeighbors=5,
    face_minSize=(30, 30),
    detect_eyes=True,
    expand_box=(0.1, 0.3)    
):

    face_cascade, eye_cascade = ensure_haarcascades(face_cascade_filename, eye_cascade_filename)

    frame_num = 0
    faces_drawn = 0
    t0 = time.perf_counter()

    try:
        total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    except Exception:
        total_frames = -1

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        frame_num += 1
        t_frame_start = time.perf_counter()

        if scale != 1.0:
            fx = float(scale)
            fy = float(scale)
            if use_cuda_resize:
                try:
                    gpu_frame = cv2.cuda_GpuMat()
                    gpu_frame.upload(frame)
                    gpu_small = cv2.cuda.resize(gpu_frame, (0, 0), fx=fx, fy=fy)
                    work_frame = gpu_small.download()
                except Exception:
                    work_frame = cv2.resize(frame, (0, 0), fx=fx, fy=fy, interpolation=cv2.INTER_LINEAR)
            else:
                work_frame = cv2.resize(frame, (0, 0), fx=fx, fy=fy, interpolation=cv2.INTER_LINEAR)
        else:
            fx = fy = 1.0
            work_frame = frame

        gray = cv2.cvtColor(work_frame, cv2.COLOR_BGR2GRAY)
        faces_small = face_cascade.detectMultiScale(
            gray,
            scaleFactor=face_scaleFactor,
            minNeighbors=face_minNeighbors,
            minSize=(max(1, int(face_minSize[0]*fx)), max(1, int(face_minSize[1]*fy)))
        )

        dx_frac, dy_frac = expand_box

        for (x, y, w, h) in faces_small:
            X = int(round(x / fx))
            Y = int(round(y / fy))
            W = int(round(w / fx))
            H = int(round(h / fy))

            dx = int(round(dx_frac * W))
            dy = int(round(dy_frac * H))
            left   = max(0, X - dx)
            top    = max(0, Y - dy)
            right  = min(frame.shape[1], X + W + dx)
            bottom = min(frame.shape[0], Y + H + dy)

            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            faces_drawn += 1

            if detect_eyes:
                roi_gray_small = gray[y:y+h, x:x+w]
                eyes_small = eye_cascade.detectMultiScale(roi_gray_small)

                for (ex, ey, ew, eh) in eyes_small:
                    ex_small = x + ex
                    ey_small = y + ey
                    EX = int(round(ex_small / fx))
                    EY = int(round(ey_small / fy))
                    EW = int(round(ew / fx))
                    EH = int(round(eh / fy))

                
                    EX2 = min(frame.shape[1], EX + EW)
                    EY2 = min(frame.shape[0], EY + EH)
                    EX  = max(0, EX)
                    EY  = max(0, EY)

                    cv2.rectangle(frame, (EX, EY), (EX2, EY2), (255, 0, 0), 2)

        out.write(frame)

        ms_per_frame = (time.perf_counter() - t_frame_start) * 1000.0
        if print_every and (frame_num % print_every == 0):
            h, w = frame.shape[:2]
            total_txt = total_frames if total_frames > 0 else "?"
            print(f"video 1/1 (frame {frame_num}/{total_txt}) [{input_video_path}]: {w}x{h} | {ms_per_frame:.1f} ms/frame")

    elapsed = time.perf_counter() - t0
    avg_ms = (elapsed / frame_num * 1000.0) if frame_num > 0 else None
    return {
        "frames_processed": frame_num,
        "faces_drawn": faces_drawn,
        "elapsed_sec": elapsed,
        "avg_ms_per_frame": avg_ms,
        "scale": scale,
        "used_cuda_resize": bool(use_cuda_resize),
        "output_path": getattr(out, 'filename', 'output file')
    }




# bad haar


In [ ]:

input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV"
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "bad_light_haarcascade_OUTPUT.mp4")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError(f"Не удалось открыть входное видео: {input_video_path}")

pos0 = video_capture.get(cv2.CAP_PROP_POS_FRAMES)
ok_probe, probe = video_capture.read()
if not ok_probe or probe is None:
    video_capture.release()
    raise RuntimeError("Не удалось прочитать первый кадр из входного видео для определения размера")
h0, w0 = probe.shape[:2]
video_capture.set(cv2.CAP_PROP_POS_FRAMES, pos0)

fps = video_capture.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 1e-3:
    fps = 25.0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (w0, h0))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

summary = run_opencv_haarcascade_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,
    use_cuda_resize=True,
    scale=0.75,
    print_every=5,
    face_cascade_filename="haarcascade_frontalface_default.xml",
    eye_cascade_filename="haarcascade_eye.xml",
    face_scaleFactor=1.1,
    face_minNeighbors=5,
    face_minSize=(30, 30),
    detect_eyes=True,
    expand_box=(0.1, 0.3)
)

video_capture.release()
out.release()

print("Готово. Результат:", summary)
print(f"Обработка завершена. Видео сохранено в {output_video_path}")

video 1/1 (frame 5/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 848.9 ms/frame
video 1/1 (frame 10/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 729.5 ms/frame
video 1/1 (frame 15/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 674.1 ms/frame
video 1/1 (frame 20/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 794.2 ms/frame
video 1/1 (frame 25/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 719.9 ms/frame
video 1/1 (frame 30/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 641.8 ms/frame
video 1/1 (frame 35/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 735.9 ms/frame
video 1/1 (frame 40/780) [/kaggle/input/face-models-data/face_model_videos/videos/bad_light.MOV]: 3840x2160 | 770.8 ms/frame
v

# good haar

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "good_light_haarcascade_OUTPUT.mp4")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError(f"Не удалось открыть входное видео: {input_video_path}")

pos0 = video_capture.get(cv2.CAP_PROP_POS_FRAMES)
ok_probe, probe = video_capture.read()
if not ok_probe or probe is None:
    video_capture.release()
    raise RuntimeError("Не удалось прочитать первый кадр из входного видео для определения размера")
h0, w0 = probe.shape[:2]

video_capture.set(cv2.CAP_PROP_POS_FRAMES, pos0)

fps = video_capture.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 1e-3:
    fps = 25.0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (w0, h0))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

summary = run_opencv_haarcascade_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,
    use_cuda_resize=True,      
    scale=0.75,               
    print_every=5,
    face_cascade_filename="haarcascade_frontalface_default.xml",
    eye_cascade_filename="haarcascade_eye.xml",
    face_scaleFactor=1.1,
    face_minNeighbors=5,
    face_minSize=(30, 30),
    detect_eyes=True,
    expand_box=(0.1, 0.3)
)



print("Готово. Результат:", summary)
print(f"Обработка завершена. Видео сохранено в {output_video_path}")


video 1/1 (frame 5/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 828.3 ms/frame
video 1/1 (frame 10/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 803.5 ms/frame
video 1/1 (frame 15/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 808.7 ms/frame
video 1/1 (frame 20/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 813.2 ms/frame
video 1/1 (frame 25/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 794.3 ms/frame
video 1/1 (frame 30/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 813.5 ms/frame
video 1/1 (frame 35/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 815.1 ms/frame
video 1/1 (frame 40/1673) [/kaggle/input/face-models-data/face_model_videos/videos/good_light.MOV]: 3840x2160 | 

# rad haar

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "red_light_haarcascade_OUTPUT.mp4")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError(f"Не удалось открыть входное видео: {input_video_path}")

pos0 = video_capture.get(cv2.CAP_PROP_POS_FRAMES)
ok_probe, probe = video_capture.read()
if not ok_probe or probe is None:
    video_capture.release()
    raise RuntimeError("Не удалось прочитать первый кадр из входного видео для определения размера")
h0, w0 = probe.shape[:2]

video_capture.set(cv2.CAP_PROP_POS_FRAMES, pos0)

fps = video_capture.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 1e-3:
    fps = 25.0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (w0, h0))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")

summary = run_opencv_haarcascade_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,
    use_cuda_resize=True,     
    scale=0.75,                
    print_every=5,
    face_cascade_filename="haarcascade_frontalface_default.xml",
    eye_cascade_filename="haarcascade_eye.xml",
    face_scaleFactor=1.1,
    face_minNeighbors=5,
    face_minSize=(30, 30),
    detect_eyes=True,
    expand_box=(0.1, 0.3)
)



print("Готово. Результат:", summary)
print(f"Обработка завершена. Видео сохранено в {output_video_path}")


video 1/1 (frame 5/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 753.6 ms/frame
video 1/1 (frame 10/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 764.1 ms/frame
video 1/1 (frame 15/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 789.3 ms/frame
video 1/1 (frame 20/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 766.8 ms/frame
video 1/1 (frame 25/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 722.3 ms/frame
video 1/1 (frame 30/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 701.2 ms/frame
video 1/1 (frame 35/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 695.0 ms/frame
video 1/1 (frame 40/762) [/kaggle/input/face-models-data/face_model_videos/videos/red_light.MOV]: 3840x2160 | 724.6 ms/frame
v

# blue haar

In [ ]:
input_video_path = "/kaggle/input/face-models-data/face_model_videos/videos/blue_light.MOV"
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "blue_light_haarcascade_OUTPUT_FINAL.mp4")

video_capture = cv2.VideoCapture(input_video_path)
if not video_capture.isOpened():
    raise RuntimeError(f"Не удалось открыть входное видео: {input_video_path}")

pos0 = video_capture.get(cv2.CAP_PROP_POS_FRAMES)
ok_probe, probe = video_capture.read()
if not ok_probe or probe is None:
    video_capture.release()
    raise RuntimeError("Не удалось прочитать первый кадр из входного видео для определения размера")
h0, w0 = probe.shape[:2]

video_capture.set(cv2.CAP_PROP_POS_FRAMES, pos0)

fps = video_capture.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 1e-3:
    fps = 25.0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (w0, h0))
if not out.isOpened():
    video_capture.release()
    raise RuntimeError("Ошибка: Не удалось открыть файл для записи.")
summary = run_opencv_haarcascade_pipeline(
    video_capture=video_capture,
    out=out,
    input_video_path=input_video_path,
    use_cuda_resize=True,      
    scale=0.75,                
    print_every=5,
    face_cascade_filename="haarcascade_frontalface_default.xml",
    eye_cascade_filename="haarcascade_eye.xml",
    face_scaleFactor=1.1,
    face_minNeighbors=5,
    face_minSize=(30, 30),
    detect_eyes=True,
    expand_box=(0.1, 0.3)      
)



print("Готово. Результат:", summary)
print(f"Обработка завершена. Видео сохранено в {output_video_path}")